**Zonal Statistics, Trends & Correlation:**
Computes per-Quartier zonal statistics (mean LST and NDVI) for each timestep. Calculates long-term warming trends using linear regression and the Mann-Kendall test. Analyses the spatial correlation between LST and NDVI across Quartiere and time using Pearson correlation.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import xarray as xr
import rioxarray
import geopandas as gpd
from scipy import stats
import pymannkendall as mk
import matplotlib.pyplot as plt

**1.** Load the processed Cube and the Statistische Quartiere shapefile

In [15]:
PROCESSED = Path("../data/processed/")

ds_cube = xr.open_dataset(PROCESSED / "02_zurich_west_cube.nc")

shapefile_path = ("../data/vector/statistische_Quartiere.gpkg")
zurich_west_kreise  = [3, 4, 5, 9, 10]

gdf_all = gpd.read_file(
    shapefile_path,
    layer="stzh.adm_statistische_quartiere_map")

gdf_zw = gdf_all[gdf_all["knr"].isin(zurich_west_kreise)].to_crs("EPSG:2056").copy()

print(ds_cube)
print(f"\nTime steps: {len(ds_cube.time)}")
print(f"Quartiere:  {len(gdf_zw)}")
print(gdf_zw[["qname", "kname"]].head(10))  # adjust column name to your data




<xarray.Dataset> Size: 7MB
Dimensions:      (time: 14, y: 240, x: 178)
Coordinates:
  * time         (time) datetime64[ns] 112B 1985-07-01 1988-07-01 ... 2024-07-01
  * y            (y) float64 2kB 1.253e+06 1.253e+06 ... 1.246e+06 1.246e+06
  * x            (x) float64 1kB 2.678e+06 2.678e+06 ... 2.683e+06 2.683e+06
    band         int64 8B ...
Data variables:
    spatial_ref  int64 8B ...
    LST          (time, y, x) float32 2MB ...
    LST_Celsius  (time, y, x) float32 2MB ...
    NDVI         (time, y, x) float32 2MB ...

Time steps: 14
Quartiere:  12
            qname     kname
4            Werd   Kreis 4
5        Sihlfeld   Kreis 3
6     Albisrieden   Kreis 9
10          Höngg  Kreis 10
15  Gewerbeschule   Kreis 5
16    Langstrasse   Kreis 4
17    Escher Wyss   Kreis 5
19           Hard   Kreis 4
21      Wipkingen  Kreis 10
28    Friesenberg   Kreis 3


**2.** Compute zonal statisics per Quartier